In [ ]:
# =========================
# Cell 1 — Setup
# =========================
import os
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.options.mode.chained_assignment = None  # avoid noisy warnings


def df_mem_gb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024**3)


def basic_report(df: pd.DataFrame, name: str, head: int = 3):
    print(f"[{name}] shape={df.shape}  mem={df_mem_gb(df):.3f} GB")
    display(df.head(head))


def missing_report(df: pd.DataFrame, cols=None, topn=30):
    if cols is None:
        cols = df.columns
    s = df[cols].isna().mean().sort_values(ascending=False)
    out = pd.DataFrame({"missing_frac": s, "missing_cnt": (s * len(df)).round().astype("int64")})
    display(out.head(topn))


# ---- Set your Kaggle dataset folder here ----
# It should contain: train.csv, building_metadata.csv, weather_train.csv
DATA_DIR = Path("E:\\repos\\LLM_traffic_query\\tests\\energy prediction\\dataset\\ashrae-energy-prediction")
assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"

TRAIN_PATH = DATA_DIR / "train.csv"
BMETA_PATH = DATA_DIR / "building_metadata.csv"
WEATHER_PATH = DATA_DIR / "weather_train.csv"

for p in [TRAIN_PATH, BMETA_PATH, WEATHER_PATH]:
    assert p.exists(), f"Missing file: {p}"

OUT_DIR = DATA_DIR / "processed_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("OUT_DIR :", OUT_DIR.resolve())

DATA_DIR: E:\repos\LLM_traffic_query\tests\energy prediction\dataset\ashrae-energy-prediction
OUT_DIR : E:\repos\LLM_traffic_query\tests\energy prediction\dataset\ashrae-energy-prediction\processed_clean


In [2]:
# =========================
# Cell 2 — Load building metadata (light cleaning only)
# =========================
bmeta_dtypes = {
    "site_id": "uint8",
    "building_id": "uint16",
    "primary_use": "category",
    "square_feet": "int32",
    "year_built": "float32",
    "floor_count": "float32",
}

bmeta = pd.read_csv(BMETA_PATH, dtype=bmeta_dtypes)

# conservative validity range (data-only sanitation): invalid years -> NaN
# (BDG2 uses 1900–2018 validity range; we keep it consistent)
bmeta.loc[
    (bmeta["year_built"].notna()) & ((bmeta["year_built"] < 1900) | (bmeta["year_built"] > 2018)), "year_built"
] = np.nan

basic_report(bmeta, "building_metadata")
print("Missingness in building metadata:")
missing_report(bmeta)

[building_metadata] shape=(1449, 6)  mem=0.000 GB


,site_id,building_id,primary_use,square_feet,year_built,floor_count
0,0,0,Education,7432,2008.0,NaN
1,0,1,Education,2720,2004.0,NaN
2,0,2,Education,5376,1991.0,NaN


Missingness in building metadata:


,missing_frac,missing_cnt
floor_count,0.755003,1094
year_built,0.534161,774
building_id,0.000000,0
site_id,0.000000,0
square_feet,0.000000,0
primary_use,0.000000,0


In [3]:
# =========================
# Cell 3 — Load weather (pre-merge cleaning)
# =========================
weather_dtypes = {
    "site_id": "uint8",
    "air_temperature": "float32",
    "cloud_coverage": "float32",
    "dew_temperature": "float32",
    "precip_depth_1_hr": "float32",
    "sea_level_pressure": "float32",
    "wind_direction": "float32",
    "wind_speed": "float32",
}

weather = pd.read_csv(WEATHER_PATH, dtype=weather_dtypes, parse_dates=["timestamp"])
basic_report(weather, "weather_raw")

print("Weather missingness BEFORE cleaning:")
missing_report(weather)
print("Raw weather row count:", len(weather))
print(
    "Sites:",
    weather["site_id"].nunique(),
    "  Time range:",
    weather["timestamp"].min(),
    "->",
    weather["timestamp"].max(),
)

[weather_raw] shape=(139773, 9)  mem=0.005 GB


,site_id,timestamp,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2016-01-01 00:00:00,25.000000,6.0,20.0,NaN,1019.700012,0.0,0.0
1,0,2016-01-01 01:00:00,24.400000,NaN,21.1,-1.0,1020.200012,70.0,1.5
2,0,2016-01-01 02:00:00,22.799999,2.0,21.1,0.0,1020.200012,0.0,0.0


Weather missingness BEFORE cleaning:


,missing_frac,missing_cnt
cloud_coverage,0.494895,69173
precip_depth_1_hr,0.359791,50289
sea_level_pressure,0.075966,10618
wind_direction,0.044844,6268
wind_speed,0.002175,304
dew_temperature,0.000808,113
air_temperature,0.000393,55
site_id,0.000000,0
timestamp,0.000000,0


Raw weather row count: 139773
Sites: 16   Time range: 2016-01-01 00:00:00 -> 2016-12-31 23:00:00


In [4]:
# =========================
# Cell 4 — Weather: sentinel cleanup, complete site-hour grid, deterministic imputation, then timestamp alignment
# =========================

# 1) Sentinel cleanup: precip -1 is used as missing in many public pipelines
weather.loc[weather["precip_depth_1_hr"] < 0, "precip_depth_1_hr"] = np.nan

# 2) Build full hourly timeline for the weather table (in original timestamp space)
tmin = weather["timestamp"].min()
tmax = weather["timestamp"].max()
full_hours = pd.date_range(tmin, tmax, freq="h")
expected_rows = len(full_hours) * weather["site_id"].nunique()
print("Expected full weather rows (sites * hours):", expected_rows)

cont_cols = ["air_temperature", "dew_temperature", "sea_level_pressure", "wind_speed"]
disc_cols = ["cloud_coverage", "wind_direction"]  # discrete-ish
precip_col = "precip_depth_1_hr"

pieces = []
inserted_rows_total = 0

for site_id, g in weather.groupby("site_id", sort=True):
    g = g.sort_values("timestamp").set_index("timestamp")
    before = len(g)

    g = g.reindex(full_hours)
    inserted = len(g) - before
    inserted_rows_total += inserted

    g["site_id"] = site_id

    # deterministic imputation:
    # - continuous columns: time interpolation + ffill/bfill
    g[cont_cols] = g[cont_cols].interpolate(method="time", limit_direction="both")
    g[cont_cols] = g[cont_cols].ffill().bfill()

    # - discrete-ish columns: ffill/bfill
    g[disc_cols] = g[disc_cols].ffill().bfill()

    # - precip: fill missing with 0 (conservative for "unknown precipitation depth")
    g[precip_col] = g[precip_col].fillna(0.0)

    g = g.reset_index().rename(columns={"index": "timestamp_gmt"})
    pieces.append(g)

weather_full = pd.concat(pieces, ignore_index=True)
print("Inserted missing site-hour rows total:", inserted_rows_total)
basic_report(weather_full, "weather_full_gmt")

print("Weather missingness AFTER imputation:")
missing_report(weather_full)

# 3) Timestamp alignment (site_id → hour shift) before merge
# Source mapping: Team Energetic Engineering preprocessing note
timediff = {0: 4, 1: 0, 2: 7, 3: 4, 4: 7, 5: 0, 6: 4, 7: 4, 8: 4, 9: 5, 10: 7, 11: 4, 12: 0, 13: 5, 14: 4, 15: 4}
weather_full["time_diff_hours"] = weather_full["site_id"].map(timediff).astype("int16")

# Align weather to local timestamps used in train:
weather_full["timestamp"] = weather_full["timestamp_gmt"] - pd.to_timedelta(weather_full["time_diff_hours"], unit="h")

# Keep only merge-relevant weather columns (+ audit columns)
weather_full = weather_full[
    [
        "site_id",
        "timestamp",
        "timestamp_gmt",
        "time_diff_hours",
        "air_temperature",
        "cloud_coverage",
        "dew_temperature",
        "precip_depth_1_hr",
        "sea_level_pressure",
        "wind_direction",
        "wind_speed",
    ]
].copy()

basic_report(weather_full, "weather_aligned_for_merge")
print("Aligned weather time range:", weather_full["timestamp"].min(), "->", weather_full["timestamp"].max())

Expected full weather rows (sites * hours): 140544
Inserted missing site-hour rows total: 771
[weather_full_gmt] shape=(140544, 9)  mem=0.006 GB


,timestamp_gmt,site_id,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,2016-01-01 00:00:00,0,25.000000,6.0,20.0,0.0,1019.700012,0.0,0.0
1,2016-01-01 01:00:00,0,24.400000,6.0,21.1,0.0,1020.200012,70.0,1.5
2,2016-01-01 02:00:00,0,22.799999,2.0,21.1,0.0,1020.200012,0.0,0.0


Weather missingness AFTER imputation:


,missing_frac,missing_cnt
cloud_coverage,0.1250,17568
sea_level_pressure,0.0625,8784
timestamp_gmt,0.0000,0
air_temperature,0.0000,0
site_id,0.0000,0
dew_temperature,0.0000,0
precip_depth_1_hr,0.0000,0
wind_direction,0.0000,0
wind_speed,0.0000,0


[weather_aligned_for_merge] shape=(140544, 11)  mem=0.007 GB


,site_id,timestamp,timestamp_gmt,time_diff_hours,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2015-12-31 20:00:00,2016-01-01 00:00:00,4,25.000000,6.0,20.0,0.0,1019.700012,0.0,0.0
1,0,2015-12-31 21:00:00,2016-01-01 01:00:00,4,24.400000,6.0,21.1,0.0,1020.200012,70.0,1.5
2,0,2015-12-31 22:00:00,2016-01-01 02:00:00,4,22.799999,2.0,21.1,0.0,1020.200012,0.0,0.0


Aligned weather time range: 2015-12-31 17:00:00 -> 2016-12-31 23:00:00


In [ ]:
# =========================
# Cell 5 — Load train (meter data) + initial sanitation
# =========================
train_dtypes = {
    "building_id": "uint16",
    "meter": "uint8",
    "meter_reading": "float32",
}

train = pd.read_csv(TRAIN_PATH, dtype=train_dtypes, parse_dates=["timestamp"])
basic_report(train, "train_raw")

# Basic sanity
print("Train time range:", train["timestamp"].min(), "->", train["timestamp"].max())
print("Any NaN meter_reading:", train["meter_reading"].isna().any())
print("Negative meter_reading count:", int((train["meter_reading"] < 0).sum()))

# Drop negative readings (conservative invalid-value removal)
neg_mask = train["meter_reading"] < 0
n_neg = int(neg_mask.sum())
if n_neg > 0:
    train = train.loc[~neg_mask].copy()
print("Dropped negative readings:", n_neg, "  Remaining rows:", len(train))

# Drop exact duplicate keys (building_id, meter, timestamp)
dup_mask = train.duplicated(subset=["building_id", "meter", "timestamp"], keep="first")
n_dup = int(dup_mask.sum())
if n_dup > 0:
    train = train.loc[~dup_mask].copy()
print("Dropped duplicate key rows:", n_dup, "  Remaining rows:", len(train))

# Create compact IDs for later group operations
t0 = train["timestamp"].min()
train["ts_idx"] = ((train["timestamp"] - t0) / np.timedelta64(1, "h")).astype("int32")
train["pair_id"] = (train["building_id"].astype("int32") * 4 + train["meter"].astype("int32")).astype("int32")

print("ts_idx range:", int(train["ts_idx"].min()), "->", int(train["ts_idx"].max()))
n_hours_total = int(train["ts_idx"].max() - train["ts_idx"].min() + 1)
print("Total expected hours in train range:", n_hours_total)

[train_raw] shape=(20216100, 4)  mem=0.282 GB


,building_id,meter,timestamp,meter_reading
0,0,0,2016-01-01,0.0
1,1,0,2016-01-01,0.0
2,2,0,2016-01-01,0.0


Train time range: 2016-01-01 00:00:00 -> 2016-12-31 23:00:00
Any NaN meter_reading: False
Negative meter_reading count: 0
Dropped negative readings: 0   Remaining rows: 20216100
Dropped duplicate key rows: 0   Remaining rows: 20216100
ts_idx range: 0 -> 8783
Total expected hours in train range: 8784


In [6]:
# =========================
# Cell 6 — Conservative meter-level drops:
#   (A) >50% timestamps missing per (building_id, meter)
#   (B) longest consecutive missing run >100 days (2400 hours)
# =========================

drop_stats = []


def record_drop(reason: str, n_rows: int):
    drop_stats.append({"reason": reason, "rows_dropped": int(n_rows)})


# (A) missing timestamp coverage
counts = train.groupby("pair_id", sort=False).size().astype("int32")
missing_ratio = 1.0 - (counts / n_hours_total)
bad_pairs_missing = set(missing_ratio[missing_ratio > 0.50].index.tolist())
print("Bad pairs with >50% timestamps missing:", len(bad_pairs_missing))

mask_bad_missing = train["pair_id"].isin(bad_pairs_missing)
n_drop = int(mask_bad_missing.sum())
train = train.loc[~mask_bad_missing].copy()
record_drop("drop_pairs_missing_ratio_gt_0.50", n_drop)
print("Rows dropped (missing>50% pairs):", n_drop, " Remaining:", len(train))

# (B) longest consecutive missing run via max gap in sorted ts_idx
# Gap computation: if ts jumps by d hours, then (d-1) hours are missing in between.
train_sorted = train.sort_values(["pair_id", "ts_idx"], kind="mergesort").reset_index(drop=True)

pair = train_sorted["pair_id"]
ts = train_sorted["ts_idx"]

same_pair = pair.eq(pair.shift(1))
diff = ts.diff()
gap_missing = (
    (diff.where(same_pair, 1) - 1).fillna(0).astype("int32")
)  # missing hours between consecutive observed points

max_gap = gap_missing.groupby(pair, sort=False).max()
bad_pairs_gap = set(max_gap[max_gap >= 2400].index.tolist())  # 100 days * 24h = 2400h
print("Bad pairs with max missing gap >= 2400 hours:", len(bad_pairs_gap))

mask_bad_gap = train["pair_id"].isin(bad_pairs_gap)
n_drop = int(mask_bad_gap.sum())
train = train.loc[~mask_bad_gap].copy()
record_drop("drop_pairs_max_gap_ge_2400h", n_drop)
print("Rows dropped (max-gap pairs):", n_drop, " Remaining:", len(train))

# Rebuild sorted view if needed later
train_sorted = None

Bad pairs with >50% timestamps missing: 18
Rows dropped (missing>50% pairs): 43566  Remaining: 20172534
Bad pairs with max missing gap >= 2400 hours: 3
Rows dropped (max-gap pairs): 16745  Remaining: 20155789


In [7]:
# =========================
# Cell 7 — Merge: train + building_metadata + weather (aligned timestamps)
# =========================
# Merge building metadata first (gives site_id for weather join)
df = train.merge(bmeta, on="building_id", how="left", validate="many_to_one")

# Now merge weather on (site_id, timestamp)
df = df.merge(weather_full, on=["site_id", "timestamp"], how="left", validate="many_to_one")

basic_report(df, "merged_df", head=5)

print("Missingness after merge (key fields):")
missing_report(
    df,
    cols=[
        "site_id",
        "primary_use",
        "square_feet",
        "air_temperature",
        "dew_temperature",
        "precip_depth_1_hr",
        "sea_level_pressure",
        "wind_speed",
        "cloud_coverage",
        "wind_direction",
    ],
)

[merged_df] shape=(20155789, 20)  mem=1.520 GB


,building_id,meter,timestamp,meter_reading,ts_idx,pair_id,site_id,primary_use,square_feet,year_built,floor_count,timestamp_gmt,time_diff_hours,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,0,2016-01-01,0.0,0,0,0,Education,7432,2008.0,NaN,2016-01-01 04:00:00,4.0,20.0,2.0,20.0,0.0,1020.0,250.0,2.6
1,1,0,2016-01-01,0.0,0,4,0,Education,2720,2004.0,NaN,2016-01-01 04:00:00,4.0,20.0,2.0,20.0,0.0,1020.0,250.0,2.6
2,2,0,2016-01-01,0.0,0,8,0,Education,5376,1991.0,NaN,2016-01-01 04:00:00,4.0,20.0,2.0,20.0,0.0,1020.0,250.0,2.6
3,3,0,2016-01-01,0.0,0,12,0,Education,23685,2002.0,NaN,2016-01-01 04:00:00,4.0,20.0,2.0,20.0,0.0,1020.0,250.0,2.6
4,4,0,2016-01-01,0.0,0,16,0,Education,116607,1975.0,NaN,2016-01-01 04:00:00,4.0,20.0,2.0,20.0,0.0,1020.0,250.0,2.6


Missingness after merge (key fields):


,missing_frac,missing_cnt
sea_level_pressure,0.039308,792285
cloud_coverage,0.024629,496425
wind_speed,0.000521,10509
precip_depth_1_hr,0.000521,10509
air_temperature,0.000521,10509
dew_temperature,0.000521,10509
wind_direction,0.000521,10509
site_id,0.000000,0
primary_use,0.000000,0
square_feet,0.000000,0


In [8]:
# =========================
# Cell 8 — Deterministic correction + conservative row drops (documented impurities)
# =========================

# (1) Correct unit mismatch for site 0 electricity meter (meter==0) using kBTU->kWh factor
# (kBTU to kWh conversion: 0.293071)
unit_fix_mask = (df["site_id"] == 0) & (df["meter"] == 0)
n_unit_fix = int(unit_fix_mask.sum())
df.loc[unit_fix_mask, "meter_reading"] = df.loc[unit_fix_mask, "meter_reading"] * np.float32(0.293071)
print("Applied unit conversion to rows:", n_unit_fix)

# (2) Drop rows missing critical fields (post-merge)
# Critical: square_feet, air_temperature, meter_reading
crit_mask = df["square_feet"].isna() | df["air_temperature"].isna() | df["meter_reading"].isna()
n_drop = int(crit_mask.sum())
df = df.loc[~crit_mask].copy()
record_drop("drop_rows_missing_critical(square_feet|air_temperature|meter_reading)", n_drop)
print("Dropped rows missing critical fields:", n_drop, " Remaining:", len(df))

# (3) Drop known early-year zero block subset (site 0, meter 0, building_id<=104, before 2016-05-21)
zero_block_mask = (
    (df["site_id"] == 0)
    & (df["meter"] == 0)
    & (df["building_id"] <= 104)
    & (df["timestamp"] < pd.Timestamp("2016-05-21"))
)
n_drop = int(zero_block_mask.sum())
df = df.loc[~zero_block_mask].copy()
record_drop("drop_site0_meter0_building<=104_before_2016-05-21", n_drop)
print("Dropped site0 early zero-block rows:", n_drop, " Remaining:", len(df))

# (4) Drop well-known extreme outlier buildings
outlier_mask = df["building_id"].isin([1099, 778])
n_drop = int(outlier_mask.sum())
df = df.loc[~outlier_mask].copy()
record_drop("drop_outlier_buildings(1099,778)", n_drop)
print("Dropped outlier building rows:", n_drop, " Remaining:", len(df))

Applied unit conversion to rows: 902913
Dropped rows missing critical fields: 10509  Remaining: 20145280
Dropped site0 early zero-block rows: 347025  Remaining: 19798255
Dropped outlier building rows: 34408  Remaining: 19763847


In [9]:
# =========================
# Cell 9 — Conservative electricity-only filter: drop (building_id, meter=0) pairs with >=30 consecutive days of zero readings
# =========================
# Implemented as run-length encoding on sorted (pair_id, ts_idx) within meter==0 subset.

df["ts_idx"] = ((df["timestamp"] - t0) / np.timedelta64(1, "h")).astype("int32")
df["pair_id"] = (df["building_id"].astype("int32") * 4 + df["meter"].astype("int32")).astype("int32")

df0 = df[df["meter"] == 0][["pair_id", "ts_idx", "meter_reading"]].copy()
print("Meter==0 subset rows:", len(df0))

df0 = df0.sort_values(["pair_id", "ts_idx"], kind="mergesort").reset_index(drop=True)
is_zero = df0["meter_reading"].values == 0.0

pair = df0["pair_id"].values
ts = df0["ts_idx"].values

# A new run starts when:
# - pair changes, OR
# - timestamp is not consecutive (+1 hour), OR
# - is_zero flag changes
pair_change = np.empty(len(df0), dtype=bool)
pair_change[0] = True
pair_change[1:] = pair[1:] != pair[:-1]

ts_break = np.empty(len(df0), dtype=bool)
ts_break[0] = True
ts_break[1:] = ts[1:] != ts[:-1] + 1

zero_change = np.empty(len(df0), dtype=bool)
zero_change[0] = True
zero_change[1:] = is_zero[1:] != is_zero[:-1]

run_start = pair_change | ts_break | zero_change
run_id = np.cumsum(run_start).astype("int32")

run_len = pd.Series(run_id).value_counts(sort=False)  # length per run_id
run_is_zero = pd.Series(is_zero).groupby(run_id).first()
run_pair = pd.Series(pair).groupby(run_id).first()

# For each pair, get max run length among zero-runs
zero_runs = pd.DataFrame({"pair_id": run_pair, "run_len": run_len, "is_zero": run_is_zero})
zero_runs = zero_runs[zero_runs["is_zero"] == True]

max_zero_run = zero_runs.groupby("pair_id")["run_len"].max()
bad_pairs_zero = set(max_zero_run[max_zero_run >= 720].index.tolist())  # 30 days * 24h = 720
print("Bad electricity pairs with >=30 days consecutive zeros:", len(bad_pairs_zero))

mask_bad_zero_pairs = (df["meter"] == 0) & (df["pair_id"].isin(bad_pairs_zero))
n_drop = int(mask_bad_zero_pairs.sum())
df = df.loc[~mask_bad_zero_pairs].copy()
record_drop("drop_meter0_pairs_consecutive_zero_run_ge_720h", n_drop)
print("Dropped rows (bad zero-run electricity pairs):", n_drop, " Remaining:", len(df))

Meter==0 subset rows: 11645142
Bad electricity pairs with >=30 days consecutive zeros: 26
Dropped rows (bad zero-run electricity pairs): 219309  Remaining: 19544538


In [10]:
# =========================
# Cell 10 — Final sanity reports
# =========================
basic_report(df, "FINAL_CLEANED_DF", head=5)

print("Final missingness snapshot (top):")
missing_report(df)

drop_df = pd.DataFrame(drop_stats)
print("Drop summary:")
display(drop_df.groupby("reason", as_index=False)["rows_dropped"].sum().sort_values("rows_dropped", ascending=False))

print("Row count before any processing:", 20216100, "(Kaggle commonly reports ~20,216,100)")
print("Row count after cleaning:", len(df))

[FINAL_CLEANED_DF] shape=(19544538, 20)  mem=1.620 GB


,building_id,meter,timestamp,meter_reading,ts_idx,pair_id,site_id,primary_use,square_feet,year_built,floor_count,timestamp_gmt,time_diff_hours,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
102,105,0,2016-01-01,23.303600,0,420,1,Education,50623,NaN,5.0,2016-01-01,0.0,3.8,0.0,2.4,0.0,1020.900024,240.0,3.1
103,106,0,2016-01-01,0.374600,0,424,1,Education,5374,NaN,4.0,2016-01-01,0.0,3.8,0.0,2.4,0.0,1020.900024,240.0,3.1
104,106,3,2016-01-01,0.000000,0,427,1,Education,5374,NaN,4.0,2016-01-01,0.0,3.8,0.0,2.4,0.0,1020.900024,240.0,3.1
105,107,0,2016-01-01,175.184006,0,428,1,Education,97532,2005.0,10.0,2016-01-01,0.0,3.8,0.0,2.4,0.0,1020.900024,240.0,3.1
106,108,0,2016-01-01,91.265297,0,432,1,Education,81580,1913.0,5.0,2016-01-01,0.0,3.8,0.0,2.4,0.0,1020.900024,240.0,3.1


Final missingness snapshot (top):


,missing_frac,missing_cnt
floor_count,0.825447,16132978
year_built,0.608706,11896883
sea_level_pressure,0.039550,772992
cloud_coverage,0.023545,460177
meter_reading,0.000000,0
timestamp,0.000000,0
meter,0.000000,0
building_id,0.000000,0
primary_use,0.000000,0
site_id,0.000000,0


Drop summary:


,reason,rows_dropped
5,drop_site0_meter0_building<=104_before_2016-05-21,347025
0,drop_meter0_pairs_consecutive_zero_run_ge_720h,219309
3,drop_pairs_missing_ratio_gt_0.50,43566
1,"drop_outlier_buildings(1099,778)",34408
2,drop_pairs_max_gap_ge_2400h,16745
4,drop_rows_missing_critical(square_feet|air_tem...,10509


Row count before any processing: 20216100 (Kaggle commonly reports ~20,216,100)
Row count after cleaning: 19544538


In [ ]:
# =========================
# Cell 11 — Save as Parquet (recommended: partitioned dataset)
# =========================
# Partitioning keeps files manageable and speeds later loading/filtering.
# This still represents one unified dataset (a parquet directory dataset).

out_path = OUT_DIR / "ashrae_train_cleaned.parquet"
print("Writing parquet dataset to:", out_path)

# You can choose either:
# A) One big file (simpler, but large):
# df.to_parquet(out_path, index=False, engine="pyarrow", compression="snappy")

# B) Partitioned dataset directory (recommended):
out_dataset_dir = OUT_DIR / "ashrae_train_cleaned_dataset"
out_dataset_dir.mkdir(exist_ok=True)

df.to_parquet(
    out_dataset_dir,
    index=False,
    engine="pyarrow",
    compression="snappy",
    partition_cols=["site_id", "meter"],
)

print("Done. Parquet dataset directory:", out_dataset_dir.resolve())

Writing parquet dataset to: E:\repos\LLM_traffic_query\tests\energy prediction\dataset\ashrae-energy-prediction\processed_clean\ashrae_train_cleaned.parquet
Done. Parquet dataset directory: E:\repos\LLM_traffic_query\tests\energy prediction\dataset\ashrae-energy-prediction\processed_clean\ashrae_train_cleaned_dataset


: 